In [0]:
spclientid = dbutils.secrets.get(scope = "adls-scope", key = "sp-client-id")
spclientsecret = dbutils.secrets.get(scope = "adls-scope", key = "sp-client-secret")
spoid = dbutils.secrets.get(scope = "adls-scope", key = "sp-tenant-id")

In [0]:
files = dbutils.fs.ls("abfss://bronze@gurboapistorage.dfs.core.windows.net/")
latest = sorted(files, key=lambda f: f.name)[-1].path
df = spark.read.json(latest)
# df.show()
# df.printSchema()
path = "abfss://silver@gurboapistorage.dfs.core.windows.net/CGY-Games"


In [0]:
from pyspark.sql.functions import explode, col, when

df_exploded = df.select(explode("games").alias("game"))
df_silver = df_exploded.select("game.id",
    "game.gameDate",
    "game.gameOutcome.lastPeriodType",
    col("game.homeTeam.id").alias("HomeTeamID"),
    col("game.homeTeam.commonName.default").alias("HomeTeamName"),
    col("game.homeTeam.score").alias("HomeTeamScore"),
    col("game.awayTeam.id").alias("AwayTeamID"),
    col("game.awayTeam.commonName.default").alias("AwayTeamName"),
    col( "game.awayTeam.score").alias("AwayTeamScore"),
    col("game.winningGoalScorer.lastName.default").alias("WinningGoalScorerLastName"))
df_silver = df_silver.withColumn(
        "CGYScore", when(col("HomeTeamName")=="Flames", col("HomeTeamScore")).otherwise(
            col("AwayTeamScore")
        )
    )
df_silver = df_silver.withColumn("OpponentScore", 
    when(col("AwayTeamName") == "Flames", col("HomeTeamScore")).otherwise(
        col("AwayTeamScore")
    )
)
df_silver = df_silver.withColumn("CGYWin", when(col("CGYScore") > col("OpponentScore"), "True").otherwise("False"))


df_silver.show()

df_silver.write.format("delta").mode("overwrite").save(path)



In [0]:
df.printSchema()